# 08 · The Inner Loops

### Recap & why now
Notebook 06 established that this vehicle tumbles if left alone, and Notebook 07 gave
us the four numbers that could stop it. What is missing is the logic that chooses them.

Attitude is the innermost, fastest and most safety-critical part of a drone's control:
lose it for a tenth of a second and the vehicle is upside down. So it gets its own two
loops, built here, before anything about position is discussed.

### Learning objectives
1. Compute an **attitude error** from two quaternions, in the body frame.
2. Apply the sign fix that makes the drone take the **short way round**.
3. Turn that error into a desired angular rate, and the rate error into a torque.
4. Feed the **gyroscopic term** forward instead of letting feedback discover it.
5. Fly the inner loops and watch a tumbling drone recover.

In [ ]:
# === Standard setup used throughout this notebook ========================
import numpy as np                 # NumPy = fast vector/matrix math, so we never hand-write loops for arithmetic.
import matplotlib.pyplot as plt     # Matplotlib is our plotting engine for every static figure below.
from matplotlib import animation   # Turns a list of frames into a playable movie (used for the animations).
from mpl_toolkits.mplot3d import Axes3D   # Registers the '3d' projection that every figure here needs.
from IPython.display import HTML    # Embeds an animation as a self-contained JS player (no ffmpeg required).

%matplotlib inline
# Render animations as an in-browser JavaScript player so they always play, on any machine.
plt.rcParams["animation.html"] = "jshtml"
# Raise the embed size cap (MB) so longer clips are not silently cut off.
plt.rcParams["animation.embed_limit"] = 60
# One consistent, readable look for every figure in the manual.
plt.rcParams.update({"figure.dpi": 80, "font.size": 11, "axes.grid": True})
# Print matrices with 3 decimals and no scientific notation, so output is easy to eyeball.
np.set_printoptions(precision=3, suppress=True)
print("Setup complete — NumPy", np.__version__, "| Matplotlib", plt.matplotlib.__version__)

In [ ]:
# === Orientation toolkit, built up over Notebooks 02-05 ==================

def quat_normalize(q):
    """Force |q| = 1. Integration drifts off the unit sphere; this pulls it back."""
    q = np.asarray(q, float)
    return q/np.linalg.norm(q)

def quat_multiply(a, b):
    """Hamilton product a (x) b: 'do b first, then a', the same reading order as matrices."""
    aw, ax, ay, az = a
    bw, bx, by, bz = b
    return np.array([aw*bw - ax*bx - ay*by - az*bz,     # Scalar part.
                     aw*bx + ax*bw + ay*bz - az*by,     # Vector part, x.
                     aw*by - ax*bz + ay*bw + az*bx,     #              y.
                     aw*bz + ax*by - ay*bx + az*bw])    #              z.

def quat_conjugate(q):
    """Flip the vector part — for a unit quaternion this is the INVERSE rotation."""
    return np.array([q[0], -q[1], -q[2], -q[3]])

def quat_to_rotmat(q):
    """The body-to-world rotation matrix that this quaternion represents."""
    w, x, y, z = quat_normalize(q)
    return np.array([[1-2*(y*y+z*z),   2*(x*y-w*z),   2*(x*z+w*y)],
                     [  2*(x*y+w*z), 1-2*(x*x+z*z),   2*(y*z-w*x)],
                     [  2*(x*z-w*y),   2*(y*z+w*x), 1-2*(x*x+y*y)]])

def euler_to_quat(roll, pitch, yaw):
    """ZYX Euler angles -> quaternion. Used to SET a pose, never to store one."""
    cr, sr = np.cos(roll/2), np.sin(roll/2)
    cp, sp = np.cos(pitch/2), np.sin(pitch/2)
    cy, sy = np.cos(yaw/2), np.sin(yaw/2)
    return np.array([cr*cp*cy + sr*sp*sy, sr*cp*cy - cr*sp*sy,
                     cr*sp*cy + sr*cp*sy, cr*cp*sy - sr*sp*cy])

def quat_to_euler(q):
    """Quaternion -> roll, pitch, yaw. For DISPLAY only — never as simulator state."""
    w, x, y, z = quat_normalize(q)
    return np.array([np.arctan2(2*(w*x + y*z), 1 - 2*(x*x + y*y)),
                     np.arcsin(np.clip(2*(w*y - z*x), -1, 1)),      # clip guards against 1+1e-16.
                     np.arctan2(2*(w*z + x*y), 1 - 2*(y*y + z*z))])

def quat_from_rotmat(R):
    """Rotation matrix -> quaternion. Four branches, so we never divide by a small number."""
    tr = np.trace(R)
    if tr > 0:
        s_ = np.sqrt(tr + 1.0)*2
        q = np.array([0.25*s_, (R[2,1]-R[1,2])/s_, (R[0,2]-R[2,0])/s_, (R[1,0]-R[0,1])/s_])
    elif R[0,0] > R[1,1] and R[0,0] > R[2,2]:
        s_ = np.sqrt(1.0 + R[0,0] - R[1,1] - R[2,2])*2
        q = np.array([(R[2,1]-R[1,2])/s_, 0.25*s_, (R[0,1]+R[1,0])/s_, (R[0,2]+R[2,0])/s_])
    elif R[1,1] > R[2,2]:
        s_ = np.sqrt(1.0 + R[1,1] - R[0,0] - R[2,2])*2
        q = np.array([(R[0,2]-R[2,0])/s_, (R[0,1]+R[1,0])/s_, 0.25*s_, (R[1,2]+R[2,1])/s_])
    else:
        s_ = np.sqrt(1.0 + R[2,2] - R[0,0] - R[1,1])*2
        q = np.array([(R[1,0]-R[0,1])/s_, (R[0,2]+R[2,0])/s_, (R[1,2]+R[2,1])/s_, 0.25*s_])
    return quat_normalize(q)

def axis_angle_to_quat(axis, angle):
    """Build a quaternion from 'rotate by `angle` about `axis`' — the geometric reading."""
    axis = np.asarray(axis, float); axis = axis/np.linalg.norm(axis)
    return np.array([np.cos(angle/2), *(axis*np.sin(angle/2))])

def quat_rotate(q, v):
    """Rotate v from the body frame into the world frame, using the sandwich product."""
    return quat_multiply(quat_multiply(q, np.array([0.0, *v])), quat_conjugate(q))[1:]

# === The vehicle, and how to draw it =====================================

PARAMS = dict(m=1.0, L=0.25,                       # Mass [kg] and hub-to-rotor distance [m].
              I=np.diag([0.01, 0.01, 0.02]),       # Inertia [kg m^2]; yaw is the heavy axis.
              d=0.016,                             # Drag torque per newton of thrust [m].
              T_min=0.0, T_max=6.0)                # What one motor can produce [N].
g = 9.81                                           # Gravity [m/s^2], along world -z.

ARM = PARAMS["L"]/np.sqrt(2)                       # Each rotor sits ARM along body x AND body y.
MOTOR_POS = np.array([[ ARM, -ARM, 0.0],           # M1 front-right.
                      [ ARM,  ARM, 0.0],           # M2 front-left.
                      [-ARM,  ARM, 0.0],           # M3 rear-left.
                      [-ARM, -ARM, 0.0]])          # M4 rear-right.
SPIN = np.array([-1.0, 1.0, -1.0, 1.0])            # +1 = counter-clockwise seen from above.

def draw_quad(ax, position, q, scale=3.0, thrusts=None):
    """Draw the drone: four arms, four rotors, a nose marker and the thrust arrow."""
    R = quat_to_rotmat(q)                          # Body-to-world, so body points become world points.
    for i, mp in enumerate(MOTOR_POS):
        tip = np.asarray(position, float) + R @ (mp*scale)
        seg = np.array([position, tip])
        ax.plot(seg[:, 0], seg[:, 1], seg[:, 2], color="0.35", lw=2)
        shade = "C3" if i in (0, 1) else "C0"      # Front rotors red, rear blue, so the nose is visible.
        if thrusts is not None:
            load = np.clip(thrusts[i]/PARAMS["T_max"], 0, 1)
            shade = plt.cm.YlOrRd(0.3 + 0.7*load)  # Colour by how hard the motor is working.
        ax.plot([tip[0]], [tip[1]], [tip[2]], "o", ms=6, color=shade)
    ax.quiver(*position, *(R[:, 2]*0.9), color="C1", lw=2.2, arrow_length_ratio=0.25)

def set_3d(ax, xlim, ylim, zlim):
    """Equal-ish 3-D axes with explicit limits, so animations do not jitter."""
    ax.set_xlim(*xlim); ax.set_ylim(*ylim); ax.set_zlim(*zlim)
    ax.set_box_aspect([xlim[1]-xlim[0], ylim[1]-ylim[0], zlim[1]-zlim[0]])
    ax.set_xlabel("x — East [m]"); ax.set_ylabel("y — North [m]"); ax.set_zlabel("z — Up [m]")

print("vehicle ready: %.1f kg, hover %.2f N total, %.3f N per motor, thrust/weight %.2f" %
      (PARAMS["m"], PARAMS["m"]*g, PARAMS["m"]*g/4, 4*PARAMS["T_max"]/(PARAMS["m"]*g)))

# === Mixing and 6-DOF dynamics, from Notebooks 06-07 =====================

POS, VEL, QUAT, OMEGA = slice(0, 3), slice(3, 6), slice(6, 10), slice(10, 13)
MIX = np.vstack([np.ones(4), MOTOR_POS[:, 1], -MOTOR_POS[:, 0], -SPIN*PARAMS["d"]])

def motor_mixer(total_thrust, torques, p=PARAMS):
    """Desired wrench -> four motor thrusts, clipped to what the hardware can do."""
    T4 = np.linalg.solve(MIX, np.array([total_thrust, *torques], float))
    return np.clip(T4, p["T_min"], p["T_max"])     # A propeller cannot pull, nor push forever.

def quad_dynamics(state, motor_thrusts, p=PARAMS, f_ext=np.zeros(3)):
    """x_dot for the 13-state quadcopter, driven by four motor thrusts."""
    q = quat_normalize(state[QUAT]); w = state[OMEGA]
    T, tx, ty, tz = MIX @ np.asarray(motor_thrusts, float)          # Geometry does its job here.
    v_dot = (quat_to_rotmat(q) @ np.array([0.0, 0.0, T])            # Thrust, body -> world.
             + np.array([0.0, 0.0, -p["m"]*g]) + f_ext)/p["m"]      # Gravity, ENU, plus any push.
    q_dot = 0.5*quat_multiply(q, np.array([0.0, *w]))               # Notebook 05's kinematics.
    w_dot = np.linalg.solve(p["I"], np.array([tx, ty, tz]) - np.cross(w, p["I"] @ w))
    return np.concatenate([state[VEL], v_dot, q_dot, w_dot])

def rk4_step(state, motor_thrusts, dt, p=PARAMS, f_ext=np.zeros(3)):
    """One RK4 step, followed by the renormalisation Notebook 05 insisted on."""
    k1 = quad_dynamics(state, motor_thrusts, p, f_ext)
    k2 = quad_dynamics(state + 0.5*dt*k1, motor_thrusts, p, f_ext)
    k3 = quad_dynamics(state + 0.5*dt*k2, motor_thrusts, p, f_ext)
    k4 = quad_dynamics(state + dt*k3, motor_thrusts, p, f_ext)
    s = state + dt/6*(k1 + 2*k2 + 2*k3 + k4)
    s[QUAT] = quat_normalize(s[QUAT])
    return s

def make_state(p=(0, 0, 0), v=(0, 0, 0), q=(1, 0, 0, 0), w=(0, 0, 0)):
    """Assemble the 13-element state vector."""
    return np.concatenate([p, v, q, w]).astype(float)

def simulate(command, T_end=4.0, dt=0.005, s0=None, p=PARAMS, f_ext=lambda t: np.zeros(3)):
    """Fly the drone. `command(t, state)` returns four motor thrusts in newtons."""
    s = make_state() if s0 is None else np.array(s0, float)
    ts, xs, ms = [0.0], [s.copy()], []
    for k in range(int(round(T_end/dt))):
        T4 = np.clip(np.asarray(command(k*dt, s), float), p["T_min"], p["T_max"])
        s = rk4_step(s, T4, dt, p, f_ext(k*dt))
        ts.append((k+1)*dt); xs.append(s.copy()); ms.append(T4)
    return np.array(ts), np.array(xs), np.array(ms)

T_HOVER = PARAMS["m"]*g                            # Total thrust that exactly cancels weight.
HOVER_EACH = T_HOVER/4                             # ...split over four identical motors.
print("model ready — hover needs %.4f N total, %.4f N per motor" % (T_HOVER, HOVER_EACH))

## 1 · The difference between two orientations

Subtracting quaternions is meaningless — they are not a vector space. The orientation
*difference* is the rotation taking you from where you are to where you want to be:

$$q_e = q^{-1} \otimes q_{\text{des}} = q^{*} \otimes q_{\text{des}}$$

(for a unit quaternion the inverse is just the conjugate). This error is expressed in
the **body** frame, which is exactly right, because the torque we are about to command
acts in the body frame.

In [ ]:
GAINS = dict(K_R=np.array([12.0, 12.0, 6.0]),      # Attitude error -> desired angular rate.
             K_w=np.array([0.06, 0.06, 0.06]))     # Angular-rate error -> torque.

def attitude_error(q, q_des):
    """Rotation from the current attitude to the desired one, in body axes."""
    q_e = quat_multiply(quat_conjugate(q), q_des)
    if q_e[0] < 0:
        q_e = -q_e                                 # The short-way-round fix; Section 2 explains it.
    return q_e

q_now = euler_to_quat(np.deg2rad(10), 0, 0)
print("  desired attitude        error quaternion        rotation needed")
for label, q_des in [("same as current   ", q_now),
                     ("level             ", np.array([1.0, 0, 0, 0])),
                     ("20° roll          ", euler_to_quat(np.deg2rad(20), 0, 0)),
                     ("30° yaw           ", euler_to_quat(0, 0, np.deg2rad(30)))]:
    q_e = attitude_error(q_now, q_des)
    ang = np.degrees(2*np.arccos(np.clip(q_e[0], -1, 1)))
    print("  %s %-24s %8.2f°" % (label, np.round(q_e, 4), ang))

print("\nFor small errors the vector part is approximately half the rotation vector, so twice it")
print("is the rotation in radians — which is why the controller below multiplies by 2.")

## 2 · One line, or the drone flips

Notebook 04's double cover comes back to bite here. $q_e$ and $-q_e$ describe the same
orientation difference, but as a **rotation** one is the short way and the other is the
long way. The scalar part tells you which: $q_{e,w} = \cos(\alpha/2)$, so a negative
scalar means $\alpha > 180°$.

> **⚠️** `if q_e[0] < 0: q_e = -q_e`. Leave it out and a small heading error near the
> wraparound commands a 350° rotation instead of a 10° one — not a wobble, a flip.

In [ ]:
def rate_command(q, q_des, K=GAINS, fix_sign=True):
    """Attitude error -> desired body angular rate, with the sign fix optional."""
    q_e = quat_multiply(quat_conjugate(q), q_des)
    if fix_sign and q_e[0] < 0:
        q_e = -q_e
    return 2.0*K["K_R"]*q_e[1:]

q_now = euler_to_quat(0, 0, np.deg2rad(170))       # Nearly at the yaw wraparound.
print("  target yaw   with the fix        without it")
for target in (175, -175, -170):
    q_des = euler_to_quat(0, 0, np.deg2rad(target))
    good = rate_command(q_now, q_des)[2]
    bad = rate_command(q_now, q_des, fix_sign=False)[2]
    print("  %8d° %14.3f rad/s %14.3f rad/s" % (target, good, bad))

print("\nRow 2 is the trap: -175° is 15° away going one way and 345° going the other. With the")
print("fix the drone commands a gentle %.2f rad/s the short way; without it, %.2f rad/s the long" %
      (abs(rate_command(q_now, euler_to_quat(0, 0, np.deg2rad(-175)))[2]),
       abs(rate_command(q_now, euler_to_quat(0, 0, np.deg2rad(-175)), fix_sign=False)[2])))
print("way round. One line of code, and it is the difference between a correction and a stunt.")

## 3 · Rate to torque

$$\tau = K_\omega\,(\omega_{\text{cmd}} - \omega) + \omega \times (I\omega)$$

A proportional controller again, plus Notebook 06's gyroscopic term added as
**feedforward**. We know that torque is coming, so we cancel it directly rather than
waiting for the feedback loop to notice an error and react.

Then $\tau$ and the thrust go to Notebook 07's mixer, and the mixer's clipping is the
last thing that happens before physics.

In [ ]:
def inner_loops(state, q_des, thrust, p=PARAMS, K=GAINS):
    """Attitude and rate loops together: desired attitude -> four motor thrusts."""
    w_cmd = rate_command(state[QUAT], q_des, K)                       # Loop 1: attitude -> rate.
    tau = K["K_w"]*(w_cmd - state[OMEGA]) + np.cross(state[OMEGA], p["I"] @ state[OMEGA])
    return motor_mixer(thrust, tau, p)                                 # Loop 2 output -> motors.

print("  body rates [rad/s]     feedforward torque [N m]   as a share of a typical command")
for w in [np.zeros(3), np.array([2.0, 0, 0]), np.array([3.0, 0, 3.0]), np.array([8.0, 6.0, 4.0])]:
    gyro = np.cross(w, PARAMS["I"] @ w)
    print("  %-22s %-26s %5.0f%%" % (np.round(w, 1), np.round(gyro, 4), 100*np.abs(gyro).max()/0.05))

print("\nIn gentle flight the term is a fraction of a percent and you could ignore it. In")
print("aggressive flight it is not, and it costs one cross product to include — which is why")
print("every real rate controller has it.")

## 4 · Recovering from a tumble

The test that matters. Throw the drone into the state Notebook 06 ended in — rolling
fast, tilted a long way over — and let the inner loops bring it back to level while the
thrust stays fixed at hover.

Nothing here knows anything about position. That is deliberate: the inner loops exist to
keep the vehicle alive, and Notebook 09 adds the layers that decide where it goes.

In [ ]:
q_level = np.array([1.0, 0.0, 0.0, 0.0])
s_bad = make_state(p=(0, 0, 5.0),
                   q=euler_to_quat(np.deg2rad(60), np.deg2rad(-40), np.deg2rad(30)),
                   w=(3.0, -2.0, 1.5))            # Tilted hard and spinning in every axis.

t, X, M = simulate(lambda t_, s_: inner_loops(s_, q_level, T_HOVER), T_end=4.0, s0=s_bad)
rpy = np.degrees(np.array([quat_to_euler(q_) for q_ in X[:, QUAT]]))
tilt = np.degrees(np.arccos(np.clip([quat_to_rotmat(q_)[2, 2] for q_ in X[:, QUAT]], -1, 1)))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.5, 3.2))
for i, lbl in enumerate(["roll", "pitch", "yaw"]):
    a1.plot(t, rpy[:, i], lw=1.7, label=lbl)
a1.set_xlabel("time [s]"); a1.set_ylabel("angle [deg]"); a1.legend(fontsize=8); a1.set_title("Attitude")
a2.plot(t, tilt, color="C3", lw=2)
a2.set_xlabel("time [s]"); a2.set_ylabel("tilt from vertical [deg]"); a2.set_title("Back to level")
plt.tight_layout(); plt.show()

settled = t[np.argmax(tilt < 2.0)]
print("started %.0f° from vertical, back inside 2° after %.2f s." % (tilt[0], settled))
print("peak motor thrust during the recovery %.2f N of %.1f available." % (M.max(), PARAMS["T_max"]))
print("The drone fell %.1f m while recovering, because holding attitude was the only job —" %
      (X[0, 2] - X[-1, 2]))
print("nothing in this loop is even looking at altitude.")

## 🧪 Try it yourself

**E1.** The attitude error is computed as $q^{*} \otimes q_{des}$ rather than
$q_{des} \otimes q^{*}$. What frame does each version live in, and why does it matter
for the torque command?

**E2.** Sweep the attitude gain $K_R$ and find where the recovery stops improving. What
goes wrong at very large values?

In [ ]:
# --- Solution E1 ---
q_a = euler_to_quat(np.deg2rad(30), 0, np.deg2rad(80))
q_b = euler_to_quat(0, 0, 0)
body_frame = quat_multiply(quat_conjugate(q_a), q_b)
world_frame = quat_multiply(q_b, quat_conjugate(q_a))
print("E1: q* (x) q_des  ->", np.round(body_frame[1:], 4), " (body frame)")
print("    q_des (x) q*  ->", np.round(world_frame[1:], 4), " (world frame)")
print("    Same rotation, different frame, and the vector parts point different ways. Our torque")
print("    is applied about BODY axes — the motors are bolted to the airframe — so the error must")
print("    be expressed in body axes too. Use the world-frame version and the drone corrects along")
print("    the wrong axes whenever it is not near level, which is precisely when it matters.")

# --- Solution E2 ---
print("\nE2:  K_R    time back inside 2°   peak rate [rad/s]   peak motor [N]")
for scale in (0.25, 0.5, 1.0, 2.0, 4.0, 8.0):
    K = {k_: (v.copy() if isinstance(v, np.ndarray) else v) for k_, v in GAINS.items()}
    K["K_R"] = GAINS["K_R"]*scale
    t2, X2, M2 = simulate(lambda t_, s_: inner_loops(s_, q_level, T_HOVER, K=K), T_end=4.0, s0=s_bad)
    tilt2 = np.degrees(np.arccos(np.clip([quat_to_rotmat(q_)[2, 2] for q_ in X2[:, QUAT]], -1, 1)))
    hit = t2[np.argmax(tilt2 < 2.0)] if np.any(tilt2 < 2.0) else np.nan
    print("    %5.1f %19.2f s %17.1f %15.2f" %
          (GAINS["K_R"][0]*scale, hit, np.abs(X2[:, OMEGA]).max(), M2.max()))
print("    Recovery keeps improving up to a point and then stalls, while the peak body rate and")
print("    the motor demand keep climbing. Past that point the extra gain is buying nothing and")
print("    spending actuator authority — and on a real vehicle, where the rate loop has delay and")
print("    the gyro has noise, it would be buying oscillation instead.")

## 🚁 Mini-project: the save

Animate the recovery. The drone starts inverted and spinning, and the two inner loops
bring it level in about a second — while it falls, because nothing here is watching the
altitude. That division of labour is the whole idea of a cascade.

In [ ]:
t, X, M = simulate(lambda t_, s_: inner_loops(s_, q_level, T_HOVER), T_end=4.0, s0=s_bad)
step = 10
fig = plt.figure(figsize=(6.6, 5.4))
ax = fig.add_subplot(111, projection="3d")

def frame(j):
    ax.clear()
    k = step*j
    ax.plot(X[:k+1, 0], X[:k+1, 1], X[:k+1, 2], color="0.75", lw=1.4)
    draw_quad(ax, X[k, POS], X[k, QUAT], scale=2.5, thrusts=M[min(k, len(M)-1)])
    set_3d(ax, (-3, 3), (-3, 3), (-4, 6))
    tilt_k = np.degrees(np.arccos(np.clip(quat_to_rotmat(X[k, QUAT])[2, 2], -1, 1)))
    ax.set_title("t = %4.2f s   tilt from vertical %5.1f°   altitude %5.2f m" %
                 (k*0.005, tilt_k, X[k, 2]), fontsize=10)
    ax.view_init(elev=18, azim=-62)
    return []

anim = animation.FuncAnimation(fig, frame, frames=len(X)//step, interval=50, blit=False)
plt.close(fig)
HTML(anim.to_jshtml())

> **🤖 Robotics connection.** The attitude and rate loops are the part of a drone that
> must never stop. On real hardware they run on the flight controller at one to eight
> kilohertz, directly off the gyroscope, and they keep running even when the companion
> computer crashes or the position estimate goes stale. If losing a higher-level process
> can tumble your aircraft, the split between the layers is in the wrong place.

**Where next.** The drone can hold an attitude. Notebook 09 puts the outer loops on top
— position and velocity — and works out which attitude to ask for in the first place.